

### ***Potential Customers Prediction Model***

**Problem Understanding**

Given that an analysis have been conducted to identify a customer who is active, relevant to the products being promoted, and meets the purchase-behavior criteria, what is the probability that they will respond to a marketing campaign or not?

**Goal:**
Optimize *marketing campaign cost efficiency* by reducing inefficient spending on uninterested customers.

**Evaluation Metric**

The model will produce two possible outcomes:

* Prediction = 1 → The model predicts that the customer will accept the offer
* Prediction = 0 → The model predicts that the customer will not accept the offer

The main focus is to accurately predict customers who will accept the *marketing campaign* offer.

In [1]:
#Impoert liberies
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

**Data Profiling**

In [2]:
#Read dataset
df = pd.read_csv(r'C:\Users\ss\OneDrive\Desktop\ML_pipeline\Data\marketing_campaign.csv',sep=';')
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0


In [3]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

**Check missing values & duplicates**

**Check data types: are there improper formating**

**Data sample: are there inconsistencies?**

In [4]:
def dataframe_observation(df):
    missing = df.isnull().sum()
    duplicate = df.duplicated().sum()
    missing_per = round(missing/len(df),4)*100
    unique_value = df.nunique()
    data_type  = df.dtypes
    sample_unique = [df[col].unique() for col in df.columns]

    df = pd.DataFrame({
        'Missing Value' : missing,
        'Percentage of Missing' : missing_per,
        'Total Duplicated Data' : duplicate,
        'Total Unique Value' : unique_value,
        'Data Type' : data_type,
        'Sample Of value' : sample_unique
    }).sort_values(by='Missing Value',ascending=False)

    return df

dataframe_observation(df)

,Missing Value,Percentage of Missing,Total Duplicated Data,Total Unique Value,Data Type,Sample Of value
Income,24,1.07,0,1974,float64,"[58138.0, 46344.0, 71613.0, 26646.0, 58293.0, ..."
ID,0,0.00,0,2240,int64,"[5524, 2174, 4141, 6182, 5324, 7446, 965, 6177..."
Year_Birth,0,0.00,0,59,int64,"[1957, 1954, 1965, 1984, 1981, 1967, 1971, 198..."
Education,0,0.00,0,5,object,"[Graduation, PhD, Master, Basic, 2n Cycle]"
Marital_Status,0,0.00,0,8,object,"[Single, Together, Married, Divorced, Widow, A..."
Kidhome,0,0.00,0,3,int64,"[0, 1, 2]"
Teenhome,0,0.00,0,3,int64,"[0, 1, 2]"
Dt_Customer,0,0.00,0,663,object,"[2012-09-04, 2014-03-08, 2013-08-21, 2014-02-1..."
Recency,0,0.00,0,100,int64,"[58, 38, 26, 94, 16, 34, 32, 19, 68, 11, 59, 8..."
MntWines,0,0.00,0,776,int64,"[635, 11, 426, 173, 520, 235, 76, 14, 28, 5, 6..."


Based on the exploration made, several findings were obtained as follows:

1. The **target variable** is `Response`, which has a value of **1** if the customer accepted the campaign offer and **0** if not.
2. 24 **Null values** found in the `Income` column, accounting for **1%** of the data. I will drop to maintain data quality.
3. Columns containing **a single unique value**, which does not contribute significantly to the model and will be removed from the dataset.
4. **Date** Column with **(object)** data type will be converted into **datetime** format.
5. No duplicate data entries were found in the dataset.

In [ ]:
# Drop Columns with a single unique value
df = df.drop(columns=['Z_CostContact','Z_Revenue'])
#Null values
df = df.dropna()
#Format the date column properly
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'],format='%Y-%m-%d')

##### Check inconsistencies in columns containing categorical data

In [6]:
#Print unique entries in categorical columns
print(df["Education"].unique())
print(df["Marital_Status"].unique()) 

['Graduation' 'PhD' 'Master' 'Basic' '2n Cycle']
['Single' 'Together' 'Married' 'Divorced' 'Widow' 'Alone' 'Absurd' 'YOLO']


**Observations**
* I assume *Alone*, *Absurd*, *Yolo* and **"Single"** are similar categories.

In [7]:
maping_marital_status = {
    'Alone':'Single',
    'Absurd':'Single',
    'YOLO':'Single'
}

df['Marital_Status'] = df['Marital_Status'].replace(maping_marital_status)


**Descriptive Statistics**

In [8]:
display(
    df.describe(),
    df.describe(include='object')
)

,ID,Year_Birth,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,...,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response
count,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216,2216.000000,2216.000000,2216.000000,2216.000000,...,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000,2216.000000
mean,5588.353339,1968.820397,52247.251354,0.441787,0.505415,2013-07-10 11:29:27.509025280,49.012635,305.091606,26.356047,166.995939,...,2.671029,5.800993,5.319043,0.073556,0.074007,0.073105,0.064079,0.013538,0.009477,0.150271
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,2012-07-30 00:00:00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2814.750000,1959.000000,35303.000000,0.000000,0.000000,2013-01-16 00:00:00,24.000000,24.000000,2.000000,16.000000,...,0.000000,3.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,2013-07-08 12:00:00,49.000000,174.500000,8.000000,68.000000,...,2.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,8421.750000,1977.000000,68522.000000,1.000000,1.000000,2013-12-31 00:00:00,74.000000,505.000000,33.000000,232.250000,...,4.000000,8.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,2014-06-29 00:00:00,99.000000,1493.000000,199.000000,1725.000000,...,28.000000,13.000000,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,3249.376275,11.985554,25173.076661,0.536896,0.544181,NaN,28.948352,337.327920,39.793917,224.283273,...,2.926734,3.250785,2.425359,0.261106,0.261842,0.260367,0.244950,0.115588,0.096907,0.357417


,Education,Marital_Status
count,2216,2216
unique,5,5
top,Graduation,Married
freq,1116,857


**Target Column Distribution**

In [9]:
df['Response'].value_counts()/len(df)*100

Response
0    84.972924
1    15.027076
Name: count, dtype: float64

The distribution of the target variable indicates a class imbalance, where only about 15% of customers accepted the campaign, while the remaining 85% did not respond positively.


**Review** 

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2216 entries, 0 to 2239
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   2216 non-null   int64         
 1   Year_Birth           2216 non-null   int64         
 2   Education            2216 non-null   object        
 3   Marital_Status       2216 non-null   object        
 4   Income               2216 non-null   float64       
 5   Kidhome              2216 non-null   int64         
 6   Teenhome             2216 non-null   int64         
 7   Dt_Customer          2216 non-null   datetime64[ns]
 8   Recency              2216 non-null   int64         
 9   MntWines             2216 non-null   int64         
 10  MntFruits            2216 non-null   int64         
 11  MntMeatProducts      2216 non-null   int64         
 12  MntFishProducts      2216 non-null   int64         
 13  MntSweetProducts     2216 non-null   i

In [11]:
#Print unique entries in categorical columns
print(df["Marital_Status"].unique())

['Single' 'Together' 'Married' 'Divorced' 'Widow']


In [ ]:
#Export data
df.to_csv(r'C:\Users\ss\OneDrive\Desktop\CUSTOMER PREDICTION\Data\cleaned_data.csv', index=False, header=True)